<a href="https://colab.research.google.com/github/BianchiLuca28/XAI-Healthcare-Task/blob/main/notebooks/Explainable.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Environment Colab Setup

In this section, we check whether the notebook as been executed on Google Colab or not.

In case it is executed on Colab, it clones the repository (which has the dataset inside) and installs the required libraries.

In [ ]:
import os

# Execute this code only if in colab
if 'COLAB_GPU' in os.environ:
  print("Executing in Colab!")
  # Cloning GitHub repository
  !git clone https://github.com/BianchiLuca28/XAI-Healthcare-Task.git
  %cd XAI-Healthcare-Task
  !pip install shap lime xgboost groq
  !pip uninstall -y scikit-learn
  !pip install scikit-learn==1.5.2

# Library imports

All required libraries are imported to then be used in the whole notebook.

Take inspirations from these libraries when implementing the next steps.

In [ ]:
# For the explanaition part
import shap
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import lime
import lime.lime_tabular
from xgboost import XGBClassifier
from sklearn.inspection import PartialDependenceDisplay
from sklearn import tree

# For the LLM
from groq import Groq

# Data loading and cleaning

The first step is loading the dataset and clean what is needed.

In [ ]:
# teaching case dataset
data = pd.read_csv("../database/02_Myocardial_infarction_complications_Database.csv")

# dataset containing the feature descriptions and possible values
feature_data = pd.read_csv("../database/feature_descriptions.csv")

These columns are removed as they refer to other complications that in this teaching case won't be used.

In [ ]:
# remove the columns corresponding to other complications that we do not want to predict
data = data.drop('KFK_BLOOD', axis=1)
data = data.drop('IBS_NASL', axis=1)
data = data.drop('LET_IS', axis=1)
data = data.drop('P_IM_STEN', axis=1)
data = data.drop('REC_IM', axis=1)
data = data.drop('DRESSLER', axis=1)
data = data.drop('RAZRIV', axis=1)
data = data.drop('OTEK_LANC', axis=1)
data = data.drop('A_V_BLOK', axis=1)
data = data.drop('FIBR_JELUD', axis=1)
data = data.drop('JELUD_TAH', axis=1)
data = data.drop('PREDS_TAH', axis=1)
data = data.drop('FIBR_PREDS', axis=1)
data = data.drop('ID', axis=1)

# remove the rows with missing values replacing them with 0
data = data.fillna(0)

In [ ]:
# we do the same for the other dataset
unwanted_names = [
    'KFK_BLOOD','IBS_NASL','LET_IS','P_IM_STEN','REC_IM',
    'DRESSLER','RAZRIV','OTEK_LANC','A_V_BLOK','FIBR_JELUD',
    'JELUD_TAH','PREDS_TAH','FIBR_PREDS','ID'
]

# Drop the rows with the values
feature_data = feature_data[~feature_data['name'].isin(unwanted_names)]

# Model training

After having preprocessed the dataset, we can use it to train the model and then perform the predictions by applying it on new samples.

Separate features and target variable.

In [ ]:
X = data.drop('ZSN', axis=1)
y = data.ZSN

Applying One-Hot Encoding to all features.

In [ ]:
one_hot_X = pd.get_dummies(X)

Splitting the dataset in train and test (using a portion of 80% train set and 20% test set)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(one_hot_X, y, test_size=0.2)

Creating a XGBoost Classifier model (with default parameters for semplicity) and fitting it using the previously defined dataset for training.

In [ ]:
model = XGBClassifier()
model.fit(X_train, y_train)

## LIME explainer

In [ ]:
# TASK: Create LIME explanation, ...

## SHAP explainer

In [ ]:
# TASK: Create SHAP summary plot, ICE plot, ...

# Creating the feature description variable

In the following code, we are going to present the name, description and possible values for each feature in a suitable manner for the llm.

We use this particular format instead of a normal json dump because of the 6000 thousand tokens per minute limit of the LLM.

In [ ]:
import json

# we convert each row to a list: [Name, Description, Possible Values]
data_as_arrays = feature_data[["name","description","possible_values"]].values.tolist()

# then we create a top-level object with "schema" and "data"
feature_metadata = {
    "schema": ["Name", "Description", "Possible Values"],
    "data": data_as_arrays
}

# lastly we dump to a JSON string with indenting
feature_descriptions = json.dumps(feature_metadata, indent=2, ensure_ascii=False)

# Result Explanation with LLM

After having created a SHAP explainer, in this section we will use a Large Language Model to try to explain what the Shap Explainer has extracted for a single patient.

Below, it is already provided the code needed to:
1. Format the prompt (but is *incomplete*).
2. Get the information of a specific patient given the index.
3. Make the call to the LLM using the formatted prompt.

### TASK

In the function *format_prompt*, what is missing is the prompt itself.

The variable **prompt** has been created already, but its content is empty. Your task is to experiment different ways to format the prompt. You will see that the more precise you are at "instructing" the LLM, the more it will be precise and provide exactly what you expect.

In [ ]:
def format_prompt(patient_data, prediction, prediction_proba):
    """
    patient_data (pd.DataFrame): The patient data.
    prediction (int): The prediction of the model for the given patient.
    prediction_proba (float): The probability of the prediction.
    """

    prompt = ""
    features_and_shap_values = ""

    # we iterate through each feature in the SHAP data
    for _, row in patient_data.iterrows():
        feature = row["Feature"]
        feature_value = row["Feature Value"]
        shap_value = row["SHAP Value"]

        # here we add the SHAP and feature information
        if isinstance(feature_value, (int, float)):
            features_and_shap_values += f"- {feature}: {feature_value:.2f} (SHAP impact: {shap_value:.2f})\n"
        else:
            features_and_shap_values += f"- {feature}: {feature_value} (SHAP impact: {shap_value:.2f})\n"

    prompt = f"""
    YOUR PROMPT HERE
    """



## Inside the prompt you could try to use patient data and the description of the features in such a way
#     prompt = f"""
# These are the patient data and shapely values:
# {features_and_shap_values}

# And these are the description of the features:
# {feature_descriptions}
# """

    return prompt

The helper function *get_patient_info()* is used to extract a dataframe containing the values of the features, and the respective shapely values, given the index of a patient.

In [ ]:
def get_patient_info(patient_index):
    """
    Extracts features, prediction, and probability for a given patient index from the SHAP values previously computed.
    """
    # Extract SHAP values for the patient
    shap_values_for_patient = vals[patient_index]

    # Get the base value (expected value of the model)
    #base_value = explainer.expected_value

    # Create a DataFrame for the patient's features and SHAP values
    patient_data = pd.DataFrame({
        'Feature': X_test.columns,
        'Feature Value': X_test.iloc[patient_index],
        'SHAP Value': shap_values_for_patient.values
    })

    # Get the model prediction and probability
    prediction = model.predict(X_test)[patient_index]
    prediction_proba = model.predict_proba(X_test)[patient_index].max()

    return patient_index, prediction, prediction_proba

Below, we extract the information about a given patient. Decide yourself which patient you want to analyze.

In [ ]:
# Replace 42 with your chosen patient
patient_index = 42

In [ ]:
patient_data, prediction, prediction_proba = get_patient_info(patient_index)

The function *llm()* takes in input the connection to the Groq client and the prompt that has to be provided to the LLM and returns the response.

In [ ]:
def llm(groq_client, prompt):
  chat_completion = groq_client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="llama-3.3-70b-versatile",
  )

  return chat_completion.choices[0].message.content

**NOTE**: Before executing this cell, be sure to replace *GROQ_API_KEY* with the provided KEY.

In [ ]:
groq_client = Groq(
    api_key="GROQ_API_KEY"
)

Providing the prompt to the LLM.

In [ ]:
print(f"Answer for patient with index {patient_index} without using Shapley Values: \n\n")
print(llm(
    groq_client,
    format_prompt(patient_data, prediction, prediction_proba)
    )
)